# Library & Data import

In [31]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [32]:
df = pd.read_csv('..\Data\\Org_W_Duplicates_data_1_0.csv', low_memory=False)
main_df = pd.read_csv('..\Data\\311_2025_Jan_Dec.csv', low_memory=False) # The dataframe before the changes we applied in FinalTable notebook

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Idan\AppData\Local\Temp\ipykernel_38840\466393794.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv('..\Data\\Org_W_Duplicates_data_1_0.csv', low_memory=False)
C:\Users\Idan\AppData\Local\Temp\ipykernel_38840\466393794.py:2: SyntaxWarning: invalid escape sequence '\D'
  main_df = pd.read_csv('..\Data\\311_2025_Jan_Dec.csv', low_memory=False) # The dataframe before the changes we applied in FinalTable notebook


In [33]:
pd.set_option('display.max_rows', 200)  # In case rows are being cut off 

# First look at the data

In [34]:
# Dataset shape
print("There are {} rows and {} columns in the dataset".format(df.shape[0], df.shape[1]))

There are 3645995 rows and 8 columns in the dataset


In [35]:
# Quick look into the first 5 rows of the dataset
df.head() 

,City,Incident Zip,Agency,Location Type,Status,Location,Resolution Time,Complaint_Type
0,WOODSIDE,11377,NYPD,Street/Sidewalk,Closed,POINT (-73.907196579197 40.743018746122),0.093333,Vehicles & Parking
1,FAR ROCKAWAY,11694,HPD,RESIDENTIAL BUILDING,Closed,POINT (-73.839021963252 40.577671995863),2.700417,Housing & Building Maintenance
2,NEW YORK,10033,HPD,RESIDENTIAL BUILDING,Closed,POINT (-73.937239002314 40.851013911318),2.400741,Housing & Building Maintenance
3,BROOKLYN,11207,HPD,RESIDENTIAL BUILDING,Closed,POINT (-73.899924654626 40.663886523056),2.555451,Housing & Building Maintenance
4,BROOKLYN,11234,DSNY,Street,Closed,POINT (-73.912414463066 40.625294042041),0.293056,Vehicles & Parking


In [36]:
# See each column and its data type
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3645995 entries, 0 to 3645994
Data columns (total 8 columns):
 #   Column           Dtype  
---  ------           -----  
 0   City             str    
 1   Incident Zip     str    
 2   Agency           str    
 3   Location Type    str    
 4   Status           str    
 5   Location         str    
 6   Resolution Time  float64
 7   Complaint_Type   str    
dtypes: float64(1), str(7)
memory usage: 222.5 MB


----=====Valid values=====----
city: Cities in new york metropolitan
Incident Zip: zip code of incident location ( Dtype is str but represents a number - will be converted later)
Agency: Service responsible for the complaint
Location Type: ( Street, Sidewalk, Residential building... )
Status: ( Closed, Open, Processing... )
Location: Cordinates of the complaint
Resolution Time: Time it took to resolve the issue
Complaint Type: ("Housing & Building Maintenance", "Vehicles & Parking"... )

In [37]:
# Lets see the problems people complain about
print(df["Complaint_Type"].unique())

<StringArray>
[            'Vehicles & Parking', 'Housing & Building Maintenance',
                          'Noise',             'Sanitation & Trash',
        'Street & Infrastructure',          'Public Order & Police',
  'Health & Environmental Safety',                     'Other/Misc',
                  'Trees & Parks',             'Construction & DOB',
       'Taxi & For-Hire Vehicles',                 'Animals & Pets']
Length: 12, dtype: str


In [38]:
df["Complaint_Type"].value_counts()

Complaint_Type
Vehicles & Parking                877554
Noise                             831176
Housing & Building Maintenance    786282
Street & Infrastructure           229255
Sanitation & Trash                224815
Public Order & Police             179686
Other/Misc                        177515
Health & Environmental Safety     141139
Trees & Parks                      71259
Construction & DOB                 63241
Animals & Pets                     34471
Taxi & For-Hire Vehicles           29602
Name: count, dtype: int64

# Duplicates 

During data remodeling, we have found out that there is a column called "Additional Details" that causes us to have duplicates. The problem with the column is that for the same complaint, we got more rows, but why?  because each row had a different additional detail. In our clustering problem we want to focus on just the complaint type without more details so we decided to remove any duplicates that appear in our data.

In [42]:
# Amount of duplicates
print(df.duplicated().sum())

363809


In [43]:
# The duplicate rows
df[df.duplicated(keep=False)]


,City,Incident Zip,Agency,Location Type,Status,Location,Resolution Time,Complaint_Type
51,NEW YORK,10075,HPD,RESIDENTIAL BUILDING,Closed,POINT (-73.957188128108 40.772507525524),59.238681,Housing & Building Maintenance
52,NEW YORK,10075,HPD,RESIDENTIAL BUILDING,Closed,POINT (-73.957188128108 40.772507525524),59.238681,Housing & Building Maintenance
53,NEW YORK,10075,HPD,RESIDENTIAL BUILDING,Closed,POINT (-73.957188128108 40.772507525524),59.238681,Housing & Building Maintenance
125,NEW YORK,10030,NYPD,Street/Sidewalk,Closed,POINT (-73.941355796357 40.81853515725),0.024942,Noise
169,BROOKLYN,11226,HPD,RESIDENTIAL BUILDING,Closed,POINT (-73.95992971344 40.644328077072),17.673727,Housing & Building Maintenance
...,...,...,...,...,...,...,...,...
3645753,BROOKLYN,11239,NYPD,Street/Sidewalk,Closed,POINT (-73.873239856683 40.654753023817),0.006181,Vehicles & Parking
3645823,BRONX,10456,NYPD,Residential Building/House,Closed,POINT (-73.903744412981 40.836940662924),0.009664,Noise
3645838,NEW YORK,10013,NYPD,Residential Building/House,Closed,POINT (-74.006742504023 40.720222323433),0.002593,Noise
3645847,BRONX,10456,NYPD,Residential Building/House,Closed,POINT (-73.903744412981 40.836940662924),0.010289,Noise


In [44]:
# Get the index of duplicated rows in df
dup_index = df[df.duplicated()].index
# Use that index to display the full rows from main_df
main_df.loc[dup_index]

,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location
52,67348640,12/31/2025 12:23:35 AM,02/28/2026 06:07:17 AM,HPD,Department of Housing Preservation and Develop...,UNSANITARY CONDITION,MOLD,NaN,RESIDENTIAL BUILDING,10075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.772508,-73.957188,POINT (-73.957188128108 40.772507525524)
53,67350742,12/31/2025 12:23:35 AM,02/28/2026 06:07:17 AM,HPD,Department of Housing Preservation and Develop...,GENERAL,CABINET,DAMAGED OR MISSING,RESIDENTIAL BUILDING,10075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.772508,-73.957188,POINT (-73.957188128108 40.772507525524)
170,67348611,12/31/2025 12:03:42 AM,01/17/2026 04:13:52 PM,HPD,Department of Housing Preservation and Develop...,PLUMBING,RADIATOR,BROKEN OR MISSING,RESIDENTIAL BUILDING,11226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.644328,-73.959930,POINT (-73.95992971344 40.644328077072)
171,67347487,12/31/2025 12:03:42 AM,01/17/2026 04:13:52 PM,HPD,Department of Housing Preservation and Develop...,DOOR/WINDOW,WINDOW FRAME,LOOSE OR DEFECTIVE,RESIDENTIAL BUILDING,11226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.644328,-73.959930,POINT (-73.95992971344 40.644328077072)
172,67348610,12/31/2025 12:03:42 AM,01/17/2026 04:13:52 PM,HPD,Department of Housing Preservation and Develop...,PLUMBING,RADIATOR,AIR VALVE BROKEN OR MISSING,RESIDENTIAL BUILDING,11226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.644328,-73.959930,POINT (-73.95992971344 40.644328077072)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3645753,63583681,01/01/2025 12:53:54 AM,01/01/2025 01:02:48 AM,NYPD,New York City Police Department,Illegal Parking,Posted Parking Sign Violation,NaN,Street/Sidewalk,11239,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.654753,-73.873240,POINT (-73.873239856683 40.654753023817)
3645823,63577671,01/01/2025 12:46:57 AM,01/01/2025 01:00:52 AM,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,10456,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.836941,-73.903744,POINT (-73.903744412981 40.836940662924)
3645838,63577686,01/01/2025 12:46:13 AM,01/01/2025 12:49:57 AM,NYPD,New York City Police Department,Noise - Residential,Loud Talking,NaN,Residential Building/House,10013,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.720222,-74.006743,POINT (-74.006742504023 40.720222323433)
3645847,63575909,01/01/2025 12:45:24 AM,01/01/2025 01:00:13 AM,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,10456,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.836941,-73.903744,POINT (-73.903744412981 40.836940662924)


Let's focus on a specific duplicate with id 170 and 172

In [45]:
main_df.loc[[170,172]]

,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location
170,67348611,12/31/2025 12:03:42 AM,01/17/2026 04:13:52 PM,HPD,Department of Housing Preservation and Develop...,PLUMBING,RADIATOR,BROKEN OR MISSING,RESIDENTIAL BUILDING,11226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.644328,-73.95993,POINT (-73.95992971344 40.644328077072)
172,67348610,12/31/2025 12:03:42 AM,01/17/2026 04:13:52 PM,HPD,Department of Housing Preservation and Develop...,PLUMBING,RADIATOR,AIR VALVE BROKEN OR MISSING,RESIDENTIAL BUILDING,11226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.644328,-73.95993,POINT (-73.95992971344 40.644328077072)


As we can see, the rows are almost identical because its the same complaint: "Plumbing" but why did it create 2 rows? Because of "Additional Details" column! We have got 2 different details for the same problem. As we said in the beginning we will remove all duplicates. ( 363809 rows which is 10% of the data)

In [47]:
# Dropping duplicates
df = df.drop_duplicates(keep=False)
print(df.duplicated().sum())

0


In [48]:
# Let's save the table as csv
df.to_csv("Org_data_2_0.csv", index=False)

# Missing Values

In [40]:
# Number of missing value for each feature.
df.isnull().sum()

City               155664
Incident Zip        29813
Agency                  0
Location Type      429725
Status                  0
Location            50024
Resolution Time     72269
Complaint_Type          0
dtype: int64

In [41]:
# Percentage of missing values for each feature.
df.isnull().mean()*100

City                4.269452
Incident Zip        0.817692
Agency              0.000000
Location Type      11.786220
Status              0.000000
Location            1.372026
Resolution Time     1.982148
Complaint_Type      0.000000
dtype: float64

As we can see, most of the columns have missing values. We will decide on how to deal with them later on.

This is the data which is Null/None/NaN... But what if the missing data was written as Unknown or Unavailable? Let's check the most common encodings.